In [7]:
import os

print("Current directory:", os.getcwd())

print("\nFiles in current directory:")

for file in os.listdir():
    print(file)

Current directory: /content

Files in current directory:
.config
disease_model.pkl
.ipynb_checkpoints
symptom_columns.pkl
sample_data


In [8]:
model = joblib.load("disease_model.pkl")
symptom_columns = joblib.load("symptom_columns.pkl")

In [9]:
import joblib

# Load files

model = joblib.load("disease_model.pkl")
symptom_columns = joblib.load("symptom_columns.pkl")

print("Model loaded successfully!")

print("\nNumber of symptoms:", len(symptom_columns))

print("\nFirst 10 symptoms:")

for symptom in symptom_columns[:10]:
    print(symptom)

Model loaded successfully!

Number of symptoms: 132

First 10 symptoms:
itching
skin_rash
nodal_skin_eruptions
continuous_sneezing
shivering
chills
joint_pain
stomach_pain
acidity
ulcers_on_tongue


In [10]:
!pip install fastapi nest-asyncio pyngrok uvicorn

from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

# Load model and symptoms

model = joblib.load("disease_model.pkl")
symptom_columns = joblib.load("symptom_columns.pkl")

# Simple NLP dictionary

symptom_dict = {
    "high_fever": ["fever", "high fever", "temperature"],
    "headache": ["headache", "head hurts"],
    "stomach_pain": ["stomach pain", "stomach hurts"],
    "vomiting": ["vomiting", "throwing up", "vomit"],
    "cough": ["cough", "coughing"],
    "chest_pain": ["chest pain"],
    "breathlessness": ["can't breathe", "breathlessness"]
}

# Create FastAPI app

app = FastAPI()

# Input format

class UserInput(BaseModel):
    text: str


@app.get("/")
def home():
    return {
        "message": "API is running"
    }


@app.post("/predict")
def predict(data: UserInput):

    text = data.text.lower()

    detected = []

    # Detect symptoms

    for symptom, phrases in symptom_dict.items():

        for phrase in phrases:

            if phrase in text:

                detected.append(symptom)
                break

    # Create vector

    vector = [0] * len(symptom_columns)

    for symptom in detected:

        if symptom in symptom_columns:

            index = symptom_columns.index(symptom)

            vector[index] = 1

    sample = pd.DataFrame(
        [vector],
        columns=symptom_columns
    )

    probabilities = model.predict_proba(sample)[0]

    results = sorted(
        zip(model.classes_, probabilities),
        key=lambda x: x[1],
        reverse=True
    )

    predictions = []

    for disease, prob in results[:3]:

        predictions.append(
            {
                "disease": disease,
                "probability": round(float(prob), 4)
            }
        )

    return {
        "detected_symptoms": detected,
        "predictions": predictions
    }

print("FastAPI app created successfully!")

FastAPI app created successfully!
